# Probabilistic Linear Inference in 3D

## The Continuous Problem

We consider the problem of inferring an unknown scalar field $m: \Omega \to \mathbb{R}$ on a 3D box domain $\Omega \subset \mathbb{R}^3$ from noisy observations $\tilde{\mathbf{d}} \in \mathbb{R}^{N_d}$.

The relationship between the model and the data is described by a linear forward operator:
$$
G: \mathcal{M} \to \mathcal{D}, \quad [G(m)]_i = \langle K_i, m \rangle_{\mathcal{M}} = \int_\Omega K_i(\mathbf{x})\,m(\mathbf{x})\,\mathrm{d}V
$$
where $K_i \in \mathcal{M}$ are sensitivity kernels and $\mathcal{D} \cong \mathbb{R}^{N_d}$ is the data space. The observed data is:
$$
\tilde{\mathbf{d}} = G(m) + \bm{\eta}, \qquad \bm{\eta} \sim \mathcal{N}(\mathbf{0}, \mathbf{C}_{\mathcal{D}})
$$

**Gaussian prior.** We regularise with a Gaussian prior $m \sim \mathcal{N}(m_0, C_0)$, where $C_0 = (-\alpha \Delta)^{-s}$ is the inverse generalised Laplacian — a bounded, self-adjoint, trace-class covariance operator on $L^2(\Omega)$.  Its Karhunen–Loève eigenfunctions are the Neumann eigenfunctions $\{\phi_k\}$ of $-\Delta$ with eigenvalues $\mu_k = (\alpha \lambda_k)^{-s}$.

**Posterior measure.** The Bayesian update gives $\mu_{\mathcal{M}}^{\tilde{\mathbf{d}}} = \mathcal{N}(\tilde{m}, C_{\mathcal{M}})$ where:
$$
C_{\mathcal{M}} = \bigl(G^* \mathbf{C}_{\mathcal{D}}^{-1} G + C_0^{-1}\bigr)^{-1}, \qquad
\tilde{m} = C_{\mathcal{M}}\bigl(G^* \mathbf{C}_{\mathcal{D}}^{-1} \tilde{\mathbf{d}} + C_0^{-1} m_0\bigr).
$$

**Property inference.** Let $T: \mathcal{M} \to \mathbb{R}^{N_p}$ be a second operator that extracts local volumetric averages of $m$.  The **property posterior** is:
$$
\mu_{\mathcal{P}}^{\tilde{\mathbf{d}}} = T_*\, \mu_{\mathcal{M}}^{\tilde{\mathbf{d}}} = \mathcal{N}\!\left(T(\tilde{m}),\; T C_{\mathcal{M}} T^*\right).
$$

---

**This notebook demonstrates the full PLI workflow using pygeoinf3D and pygeoinf:**

1. Build forward operator $G$ and property operator $T$ from 3D volumetric kernels
2. Assign a Laplacian-based Gaussian prior on $L^2(\Omega)$, sample with KL expansion
3. Generate synthetic noisy data from a known true model
4. Run Bayesian inversion via `LinearBayesianInversion + CholeskySolver`
5. Visualise the prior, posterior mean, and property posterior with uncertainty

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pygeoinf base library — Bayesian inversion machinery
from pygeoinf import (
    EuclideanSpace,
    GaussianMeasure,
    LinearForwardProblem,
    LinearBayesianInversion,
    CholeskySolver,
)

# pygeoinf3D — 3D geometry, spaces, operators, sampling, and plotting
from pygeoinf3d import (
    BoxDomain,
    Lebesgue3D,
    Function3D,
    BoxLaplacianSpectrumProvider,
    SOLAOperator3D,
    FunctionToKernelAdapter,
    GeneralizedBoxLaplacian,
    KLSampler3D,
)
from pygeoinf3d.providers.function_providers import GaussianProvider3D, BumpProvider3D
from pygeoinf3d.plot import (
    plot_function3d_axis_slice,
    compare_function3d_axis_sliders,
)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

## 1. Domain, Spaces, and Parameters

We work on the unit cube $\Omega = [0,1]^3$ with the standard $L^2(\Omega)$ inner product.
Key parameters are collected here for easy tuning.

In [ ]:
import numpy as np

# ── domain ────────────────────────────────────────────────────────────────────
box  = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

# ── spectral / inference parameters ──────────────────────────────────────────
# SPEC_TRUNCATION=(5,5,5) → 6³=216 Fourier modes; balances fidelity and speed.
# Reduce to (4,4,4) for a quick dry-run (~125 modes, ~10 s inference).
SPEC_TRUNCATION = (5, 5, 5)    # max spectral index per axis
N_D             = 25           # number of Gaussian sensing kernels
KERNEL_SIGMA    = 0.12         # sensing kernel half-width (fraction of domain)

# ── data noise ────────────────────────────────────────────────────────────────
NOISE_FRAC = 0.05              # noise std = NOISE_FRAC × max(|d_true|)

# ── property operator T ───────────────────────────────────────────────────────
# 2×2×2 grid of bump-function properties at interior points of the domain.
PROPERTY_CENTERS = np.stack(
    np.meshgrid([0.25, 0.75], [0.25, 0.75], [0.25, 0.75]), axis=-1
).reshape(-1, 3)
PROPERTY_WIDTH = 0.25          # bump half-width

# ── prior ─────────────────────────────────────────────────────────────────────
# C₀ = (-α Δ)^{power} with Neumann BCs.  With power=-1 this is the Green's
# operator of the Robin-regularised Laplacian.  Larger α ↔ smaller variance
# ↔ stronger smoothness preference.  α=0.2 gives prior amplitude ≈ 2–5,
# which is visually comparable to the true model (range 0–1).
PRIOR_POWER  = -1
PRIOR_ALPHA  = 0.2             # regularisation strength
N_KL_MODES   = 100             # KL truncation for prior sampling
N_PRIOR_SAMPLES = 6            # number of prior samples to visualise

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print(f"Domain: {box}")
print(f"Spectral modes: {(SPEC_TRUNCATION[0]+1)**3}")
print(f"Sensing kernels: {N_D}")
print(f"Target properties: {len(PROPERTY_CENTERS)}")

## 2. True Model and Synthetic Data

The "ground truth" is a smooth model concentrated near the cube centre:
$$
\bar{m}(\mathbf{x}) = \sin(\pi x)\,\sin(\pi y)\,\sin(\pi z)
$$
We compute the noiseless responses $\bar{\mathbf{d}} = G(\bar{m})$ and add Gaussian noise.

In [ ]:
def _true_fn(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    return np.sin(np.pi * x[..., 0]) * np.sin(np.pi * x[..., 1]) * np.sin(np.pi * x[..., 2])

m_bar = Function3D(box, evaluate_callable=_true_fn, name=r"$\bar{m}$")

# Quick slice to confirm the model looks sensible
fig, ax, _ = plot_function3d_axis_slice(m_bar, axis='z', coord=0.5,
                                         resolution=60, show_plot=False,
                                         title=r"True model $\bar{m}$ — z = 0.5 slice")
plt.show()

## 3. Forward Operator $G$ and Target Operator $T$

Both operators are built as `SOLAOperator3D` — stacked volumetric inner products:

- **$G$**: $N_d$ Gaussian sensing kernels placed randomly in the domain.
- **$T$**: bump-function kernels on a $2\!\times\!2\!\times\!2$ grid that extract local spatial averages.

In [ ]:
# ── Forward operator G ───────────────────────────────────────────────────────
fwd_centers = rng.uniform(0.1, 0.9, (N_D, 3))
fwd_prov    = GaussianProvider3D(box, fwd_centers, sigmas=KERNEL_SIGMA)
fwd_kprov   = FunctionToKernelAdapter(fwd_prov, space)
D_space     = EuclideanSpace(N_D)
G           = SOLAOperator3D(space, D_space, fwd_kprov)

# ── Target operator T ────────────────────────────────────────────────────────
tgt_prov  = BumpProvider3D(box, PROPERTY_CENTERS, widths=PROPERTY_WIDTH)
tgt_kprov = FunctionToKernelAdapter(tgt_prov, space)
P_space   = EuclideanSpace(len(PROPERTY_CENTERS))
T         = SOLAOperator3D(space, P_space, tgt_kprov)

print(f"Forward operator  G : L²(Ω) → R^{D_space.dim}")
print(f"Property operator T : L²(Ω) → R^{P_space.dim}")

# Visualise one sensing kernel (index 0) and one target kernel at (0.25,0.25,0.25)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_function3d_axis_slice(
    G.kernels[0].kernel, axis='z', coord=0.5, resolution=60, show_plot=False,
    title="Sensing kernel $K_0$ (z=0.5)", ax=axes[0])
plot_function3d_axis_slice(
    T.kernels[0].kernel, axis='z', coord=0.5, resolution=60, show_plot=False,
    title="Target kernel $T_0$ (z=0.5)", ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# Generate synthetic data
d_bar   = G(m_bar)
noise_std = NOISE_FRAC * float(np.max(np.abs(d_bar)))
d_tilde  = d_bar + rng.normal(0.0, noise_std, D_space.dim)

# Data covariance (diagonal, known noise)
C_D = noise_std**2 * np.eye(D_space.dim)

print(f"Max |d_bar| = {np.max(np.abs(d_bar)):.4f}")
print(f"Noise std   = {noise_std:.4f}")
print(f"SNR ≈ {np.max(np.abs(d_bar)) / noise_std:.1f}")

# Quick scatter: true data vs noisy data
fig, ax = plt.subplots(figsize=(8, 3))
idx = np.arange(N_D)
ax.scatter(idx, d_bar,   label="True $G(\\bar{m})$",  color='tab:red',  marker='x', s=60, zorder=5)
ax.scatter(idx, d_tilde, label="Noisy $\\tilde{d}$",   color='tab:blue', alpha=0.7,  s=30)
for i in range(N_D):
    ax.plot([i, i], [d_bar[i], d_tilde[i]], color='gray', alpha=0.3, lw=0.8)
ax.set_xlabel("Observation index"); ax.set_ylabel("Value")
ax.set_title("Synthetic observations"); ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

## 4. Gaussian Prior on $L^2(\Omega)$

The prior covariance is the inverse generalised Laplacian $C_0 = (-\alpha\Delta)^s$ with $s < 0$:

$$
C_0 \phi_k = \mu_k \phi_k, \qquad \mu_k = (\alpha \lambda_k)^{s}
$$

where $\lambda_k$ are the Neumann Laplacian eigenvalues and $\alpha > 0$ sets the correlation length.  Samples are drawn via a truncated KL expansion.

**Note on `PRIOR_POWER`:** Using `power = -1` gives the standard inverse Laplacian prior (Laplace-type covariance).  Using `power = -2` gives smoother samples (Matérn-like).  This is the pygeoinf3D equivalent of `BesselSobolevInverse` in intervalinf — the Bessel–Sobolev operator $(-\alpha\Delta + \kappa^2 I)^{-s}$ (with shift) is a future extension tracked in the active plan.

In [ ]:
# Spectral basis (Neumann BCs for zero-mean-gradient fields)
spec_prov = BoxLaplacianSpectrumProvider(box, truncation=SPEC_TRUNCATION, bc_type='neumann')

# Covariance operator C₀ = (-alpha * Δ)^power  (power < 0 → bounded, trace-class)
cov_op = GeneralizedBoxLaplacian(
    space, spec_prov,
    power=PRIOR_POWER,
    alpha=PRIOR_ALPHA,
)
print(f"C₀ = (-{PRIOR_ALPHA}·Δ)^{PRIOR_POWER}   ({cov_op.n_modes} spectral modes)")

# KL sampler
kl_prior = KLSampler3D(cov_op, n_modes=N_KL_MODES)
prior_measure = GaussianMeasure(
    covariance=cov_op,
    expectation=space.zero,
    sample=kl_prior.sample,
)
print(f"KL prior: {kl_prior.n_modes} modes,"
      f"  λ_max={kl_prior.eigenvalues.max():.4f},"
      f"  λ_min={kl_prior.eigenvalues.min():.6f}")

In [ ]:
# Visualise 6 prior samples (z = 0.5 slice, same colour scale across samples)
N_PRIOR_SAMPLES = 6
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()

prior_samples = kl_prior.samples(N_PRIOR_SAMPLES)

# Determine a shared colour range from all samples
all_vals = np.concatenate([
    s(box.structured_mesh(20)).ravel() for s in prior_samples
])
vmax = float(np.percentile(np.abs(all_vals), 95))

for i, (s, ax) in enumerate(zip(prior_samples, axes)):
    plot_function3d_axis_slice(
        s, axis='z', coord=0.5, resolution=50, show_plot=False,
        title=f"Prior sample {i+1}  (z=0.5)", ax=ax,
        vmin=-vmax, vmax=vmax, cmap='RdBu_r')

plt.suptitle(r"Prior samples from $\mathcal{N}(0,\, C_0)$", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Bayesian Inference

We pass the forward problem and the prior to `LinearBayesianInversion`.  The posterior is computed by `CholeskySolver`, which factorises the Gram matrix of the data-space problem and solves for the posterior covariance factor.

In [ ]:
import time

# ── Data noise measure ────────────────────────────────────────────────────────
data_error_measure = GaussianMeasure.from_covariance_matrix(
    D_space, C_D, expectation=np.zeros(D_space.dim)
)

# ── Bayesian machinery ────────────────────────────────────────────────────────
fwd_problem = LinearForwardProblem(G, data_error_measure=data_error_measure)
bayes_inv   = LinearBayesianInversion(fwd_problem, prior_measure)
solver      = CholeskySolver()

# ── Compute posterior ─────────────────────────────────────────────────────────
t0 = time.perf_counter()
posterior = bayes_inv.model_posterior_measure(d_tilde, solver)
m_tilde   = posterior.expectation         # posterior mean (Function3D)
elapsed   = time.perf_counter() - t0

print(f"Posterior computed in {elapsed:.2f}s")
print(f"Data misfit — prior:     {np.linalg.norm(G(space.zero) - d_tilde):.4f}")
print(f"Data misfit — posterior: {np.linalg.norm(G(m_tilde) - d_tilde):.4f}")

## 6. Visualising the Posterior Mean

We compare the true model $\bar{m}$, the posterior mean $\tilde{m}$, and the residual 
$\bar{m} - \tilde{m}$ on a synced slice slider.

The posterior mean should match the gross structure of $\bar{m}$; the residual 
shows where the 25 observations provide limited constraint.

In [ ]:
# Side-by-side static slices at z=0.5
def _residual_fn(x):
    return _true_fn(x) - m_tilde(x)

residual = Function3D(box, evaluate_callable=_residual_fn, name=r"$\bar{m} - \tilde{m}$")
m_tilde.name = r"$\tilde{m}$ (posterior mean)"

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
vmax_model = 1.05  # known range of sin³ product

for ax, fn, title in zip(
    axes,
    [m_bar, m_tilde, residual],
    [r"True model $\bar{m}$", r"Posterior mean $\tilde{m}$", r"Residual $\bar{m}-\tilde{m}$"],
):
    vmax_this = vmax_model if "Residual" not in title else 0.5
    plot_function3d_axis_slice(
        fn, axis='z', coord=0.5, resolution=60, show_plot=False,
        title=title + "  (z=0.5)", ax=ax,
        vmin=-vmax_this, vmax=vmax_this, cmap='RdBu_r')

plt.tight_layout(); plt.show()

## 7. Property Posterior

The target operator $T : \mathcal{M} \to \mathbb{R}^p$ extracts spatially averaged properties. Given the posterior $\mu^{d_\epsilon}$ on model space, the **property posterior** is the push-forward measure

$$
\mu_P^{d_\epsilon} = T_\# \mu^{d_\epsilon}
$$

which is again Gaussian with mean $T\tilde{m}$ and covariance $T \tilde{C} T^\dagger$.

For each of the $p = 8$ target locations we report the posterior mean and a 2σ credibility interval.

In [ ]:
# Push forward the model posterior through T to get a Gaussian on R^p
prop_post = posterior.affine_mapping(operator=T)

# Extract true and posterior-mean property vectors (T is callable: T(fn) -> ndarray)
p_true  = T(m_bar)    # ndarray shape (p,)
p_tilde = T(m_tilde)  # ndarray shape (p,)

# Extract posterior covariance and standard deviations
C_P  = prop_post.covariance.matrix(galerkin=True, dense=True)   # (p, p)
std_P = np.sqrt(np.diag(C_P))

print("True properties :", np.round(p_true,  4))
print("Post. mean props:", np.round(p_tilde, 4))
print("Post. std  props:", np.round(std_P,   4))

# Coverage: fraction of true values inside ±2σ posterior credible interval
within_2sigma = np.abs(p_true - p_tilde) < 2 * std_P
print(f"\nFraction within ±2σ: {within_2sigma.mean()*100:.0f}%  ({within_2sigma.sum()} / {len(p_true)})")

In [ ]:
# Bar chart: true vs posterior mean ± 2σ per property location
p = len(p_true)
x = np.arange(p)
labels = [f"P{i+1}" for i in range(p)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.18, p_true,  width=0.35, label="True $T\\bar{m}$",  color='royalblue', alpha=0.8)
ax.bar(x + 0.18, p_tilde, width=0.35, label="Post. mean $T\\tilde{m}$", color='tomato',    alpha=0.8)
ax.errorbar(x + 0.18, p_tilde, yerr=2*std_P,
            fmt='none', color='black', capsize=4, linewidth=1.2, label='±2σ credible interval')
ax.axhline(0, color='gray', linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel("Property index"); ax.set_ylabel("Property value")
ax.set_title("Property posterior: true vs. posterior mean ± 2σ")
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 8. Posterior Samples

Drawing samples from $\mu^{d_\epsilon}$ provides an intuitive picture of the uncertainty in the model posterior. Each sample is a plausible 3-D field consistent with the data and the prior.

In [ ]:
N_SAMPLES = 4
fig, axes = plt.subplots(1, N_SAMPLES + 1, figsize=(4*(N_SAMPLES+1), 4))

# First column: posterior mean for reference
plot_function3d_axis_slice(
    m_tilde, axis='z', coord=0.5, resolution=60, show_plot=False,
    title=r"Posterior mean $\tilde{m}$  (z=0.5)", ax=axes[0],
    vmin=-1.05, vmax=1.05, cmap='RdBu_r')

for k in range(N_SAMPLES):
    m_s = posterior.sample()
    plot_function3d_axis_slice(
        m_s, axis='z', coord=0.5, resolution=60, show_plot=False,
        title=f"Sample {k+1}  (z=0.5)", ax=axes[k+1],
        vmin=-1.05, vmax=1.05, cmap='RdBu_r')

plt.suptitle("Posterior samples — each is a plausible 3-D model given the data", y=1.01)
plt.tight_layout(); plt.show()